# Python para Ingeniería de Datos — BSG Institute
## Sesión 10 · Bloque 3.6.x
### Visualización con Streamlit

---

**Lo que veremos hoy:**
1. ¿Qué es Streamlit y cómo funciona?
2. Anatomía de `dashboard.py` — cada sección explicada
3. Componentes de Streamlit: métricas, gráficas, tablas, filtros
4. Cómo Streamlit consume la API de la sesión 9
5. Cache de datos: por qué y cómo usarlo

---

> **Antes de ejecutar este notebook:**
> 1. **Terminal 1:** `uvicorn api:app --reload --port 8000`
> 2. **Terminal 2:** `streamlit run dashboard.py`
> 3. Verifica que el dashboard abre en http://localhost:8501
> 4. Luego regresa aquí

---
## CONCEPTO: ¿Qué es Streamlit?

Hasta ahora el pipeline tiene estas capas:

```
CSV/Generador → Python (ETL) → MySQL → FastAPI
```

La API expone los datos, pero solo quien sabe usar `httpx` o Swagger puede verlos. **Streamlit agrega la última capa:** una interfaz web que cualquier persona puede usar, sin saber programar.

```
CSV/Generador → Python (ETL) → MySQL → FastAPI → Streamlit (Dashboard)
                                                        ↑
                                              La gerencia ve esto
```

> **Analogía:** La API es la cocina y el menú técnico. Streamlit es el comedor con menú en español para el cliente final.

**Por qué Streamlit y no React, Angular, etc.:**
- Se escribe 100% en Python — sin HTML, CSS ni JavaScript
- Cada vez que cambias una variable, la página se actualiza sola
- Ideal para dashboards internos, prototipos y presentaciones de datos
- No es para producción masiva — para eso usarías React. Pero para DE es perfecto.

---
## CELDA 1 — Verificar que la API está activa

El dashboard depende de la API. Si la API está caída, el dashboard no muestra datos.
Siempre verifica el health check antes de presentar.

In [ ]:
import httpx
import pandas as pd
import json

BASE_URL = 'http://localhost:8000'

def pretty(r):
    print(f'Status: {r.status_code}')
    print(json.dumps(r.json(), indent=2, ensure_ascii=False, default=str))

r = httpx.get(f'{BASE_URL}/health')
pretty(r)

---
## CELDA 2 — Anatomía de dashboard.py

Antes de ver el dashboard en el browser, entendamos su estructura.
Un dashboard de Streamlit siempre tiene estas partes:

In [ ]:
# Lee el archivo dashboard.py y muestra su estructura
with open('dashboard.py', encoding='utf-8') as f:
    contenido = f.read()

# Extraer las secciones principales (líneas con comentarios de sección)
secciones = [l.strip() for l in contenido.split('\n') if l.strip().startswith('# ──')]

print('Estructura de dashboard.py:')
print()
for i, s in enumerate(secciones, 1):
    print(f'  {i}. {s.replace("# ──", "").strip()}')

---
## CELDA 3 — Cómo Streamlit consume la API

El dashboard usa exactamente las mismas llamadas `httpx` que aprendiste en la sesión 9.
La única diferencia es que están dentro de funciones con `@st.cache_data`.

In [ ]:
# Replicamos aquí la misma lógica que dashboard.py usa internamente

# 1. Obtener resumen general
resumen = httpx.get(f'{BASE_URL}/resumen').json()
print('Resumen general:')
for k, v in resumen.items():
    print(f'  {k}: {v}')

print()

# 2. Obtener transacciones y convertir a DataFrame
txs = httpx.get(f'{BASE_URL}/transacciones', params={'limite': 50}).json()['transacciones']
df  = pd.DataFrame(txs)
df['amount'] = df['amount'].astype(float)
df['fecha']  = pd.to_datetime(df['fecha'])

print(f'DataFrame de transacciones: {df.shape[0]} filas × {df.shape[1]} columnas')
print(df.dtypes)
print()
print(df.head(3))

---
## CELDA 4 — Cache de datos: `@st.cache_data`

Streamlit re-ejecuta **todo el script** cada vez que el usuario interactúa con un filtro o botón.
Sin cache, haría una llamada a la API en cada clic — lento y costoso.

`@st.cache_data(ttl=30)` guarda el resultado en memoria por 30 segundos.
Si el usuario hace clic 10 veces en ese tiempo, solo hace 1 llamada a la API.

In [ ]:
import time

# Simulamos lo que hace el cache: medir el tiempo de respuesta

def llamada_sin_cache():
    return httpx.get(f'{BASE_URL}/metricas/sucursales').json()

# Primera llamada — va a la API
t0 = time.time()
data1 = llamada_sin_cache()
t1 = time.time()
print(f'Primera llamada (red):   {(t1-t0)*1000:.1f} ms')

# Segunda llamada — también va a la API (sin cache)
t2 = time.time()
data2 = llamada_sin_cache()
t3 = time.time()
print(f'Segunda llamada (red):   {(t3-t2)*1000:.1f} ms')

# Con cache (diccionario simple como ejemplo)
_cache = {}
def llamada_con_cache(key):
    if key not in _cache:
        _cache[key] = httpx.get(f'{BASE_URL}/metricas/sucursales').json()
    return _cache[key]

t4 = time.time()
data3 = llamada_con_cache('sucursales')  # va a la red
t5 = time.time()
data4 = llamada_con_cache('sucursales')  # viene del cache
t6 = time.time()

print(f'\nCon cache:')
print(f'Primera llamada (red):   {(t5-t4)*1000:.1f} ms')
print(f'Segunda llamada (cache): {(t6-t5)*1000:.1f} ms  ← diferencia')

---
## CELDA 5 — Gráficas con Plotly

El dashboard usa Plotly para las gráficas. Aquí las construimos paso a paso
para entender cómo funciona antes de verlas en Streamlit.

In [ ]:
import plotly.express as px

# Datos de sucursales para graficar
suc_data = httpx.get(f'{BASE_URL}/metricas/sucursales').json()['sucursales']
df_suc   = pd.DataFrame(suc_data)

print('Datos de sucursales:')
print(df_suc.to_string())

# Gráfica de barras — ventas por sucursal
fig = px.bar(
    df_suc,
    x='sucursal',
    y='ventas_totales',
    color='ventas_totales',
    color_continuous_scale=['#2C4290', '#AFCA0A'],
    title='Ventas Totales por Sucursal (solo COMPLETADAS)',
    labels={'sucursal': 'Sucursal', 'ventas_totales': 'Ventas ($)'},
    text_auto='.2s',
)
fig.update_layout(plot_bgcolor='white', showlegend=False)
fig.show()

In [ ]:
# Gráfica de línea — ventas por mes
mes_data = httpx.get(f'{BASE_URL}/metricas/mensual').json()['meses']
df_mes   = pd.DataFrame(mes_data)
df_mes['periodo'] = df_mes['anio'].astype(str) + '-' + df_mes['mes'].astype(str).str.zfill(2)

fig2 = px.line(
    df_mes,
    x='periodo',
    y='ventas_totales',
    markers=True,
    title='Ventas Mensuales (tendencia)',
    labels={'periodo': 'Período', 'ventas_totales': 'Ventas ($)'},
    color_discrete_sequence=['#002060'],
)
fig2.update_traces(line_width=2.5, marker_size=8, marker_color='#AFCA0A')
fig2.update_layout(plot_bgcolor='white')
fig2.show()

---
## CELDA 6 — Filtros en Streamlit: cómo funcionan

Cuando el usuario mueve el slider o cambia el selectbox en el dashboard,
Streamlit llama de nuevo a la API con los nuevos parámetros.

Aquí replicamos esa lógica manualmente:

In [ ]:
# Simular los filtros del sidebar del dashboard

def obtener_transacciones_filtradas(status=None, sucursal=None, limite=20):
    params = {'limite': limite}
    if status and status != 'Todos':
        params['status'] = status
    if sucursal and sucursal != 'Todas':
        params['sucursal'] = sucursal

    r    = httpx.get(f'{BASE_URL}/transacciones', params=params)
    data = r.json()
    return pd.DataFrame(data['transacciones']), data['filtros']

# Sin filtros
df1, f1 = obtener_transacciones_filtradas(limite=5)
print(f'Sin filtros: {len(df1)} registros')
print(df1[['id_transaccion','status','store','amount']].to_string())

print()

# Con filtro de status
df2, f2 = obtener_transacciones_filtradas(status='COMPLETADA', limite=5)
print(f'Solo COMPLETADAS: {len(df2)} registros')
print(df2[['id_transaccion','status','store','amount']].to_string())

---
## CELDA 7 — Búsqueda por cliente

El dashboard tiene una sección de búsqueda por `customer_id`.
Esta usa el endpoint `/clientes/{id}` que implementaste en la **tarea de la sesión 9**.

In [ ]:
# Probar la búsqueda por cliente
# Primero obtenemos un customer_id válido de la tabla
txs_sample = httpx.get(f'{BASE_URL}/transacciones', params={'limite': 10}).json()['transacciones']
ids_validos = [t['customer_id'] for t in txs_sample if t['customer_id']]

if ids_validos:
    cid = ids_validos[0]
    print(f'Buscando customer_id: {cid}')

    r = httpx.get(f'{BASE_URL}/clientes/{cid}')
    print(f'Status: {r.status_code}')

    if r.status_code == 200:
        data = r.json()
        txs  = data.get('transacciones', [])
        df_c = pd.DataFrame(txs)
        total = df_c['amount'].astype(float).sum()
        print(f'Transacciones del cliente: {len(txs)}')
        print(f'Total compras: ${total:,.2f}')
        print(df_c[['id_transaccion','fecha','amount','status']].to_string())
    elif r.status_code == 404:
        print('Cliente no encontrado')
else:
    print('No hay customer_ids disponibles en la muestra')

---
## CELDA 8 — El pipeline completo de punta a punta

Visualicemos todo lo que construimos en el curso hasta aquí:

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║         PIPELINE COMPLETO — Python para DE · BSG            ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  [Sesión 7-8]                                                ║
║  CSV / Generador                                             ║
║       ↓                                                      ║
║  extraer()  →  transformar()  →  cargar_a_mysql()            ║
║                                →  cargar_a_parquet()         ║
║                                →  calcular_metricas()        ║
║       ↓                                                      ║
║  MySQL (Aiven)  +  Parquet (local)                           ║
║                                                              ║
║  [Sesión 9]                                                  ║
║  FastAPI (api.py)                                            ║
║    GET /health                                               ║
║    GET /resumen                                              ║
║    GET /transacciones?status=&sucursal=                      ║
║    GET /transacciones/{id}                                   ║
║    GET /metricas/sucursales                                  ║
║    GET /metricas/mensual                                     ║
║    GET /clientes/{id}          ← tarea sesión 9              ║
║    GET /metricas/top-sucursal  ← tarea sesión 9              ║
║       ↓                                                      ║
║  [Sesión 10]                                                 ║
║  Streamlit (dashboard.py)                                    ║
║    KPIs  |  Gráficas  |  Tabla filtrable  |  Buscador        ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")

---
## Resumen: ¿qué construiste hoy?

| Componente | Archivo | Concepto |
|---|---|---|
| Configuración de página | `st.set_page_config()` | Layout, título, ícono |
| Sidebar con filtros | `st.sidebar` | Inputs del usuario |
| KPIs | `st.metric()` | 4 columnas con valores clave |
| Gráfica de barras | `px.bar()` + `st.plotly_chart()` | Ventas por sucursal |
| Gráfica de línea | `px.line()` + `st.plotly_chart()` | Tendencia mensual |
| Tabla interactiva | `st.dataframe()` | Con colores por status |
| Buscador | `st.text_input()` + `st.button()` | Búsqueda por cliente |
| Cache | `@st.cache_data(ttl=30)` | Rendimiento |

---
## TAREA — Para la próxima sesión

Añade **dos secciones nuevas** al dashboard:

**Sección A — Gráfica de pastel por status**
- Usa `px.pie()` con los datos de `/resumen` o `/transacciones`
- Muestra qué porcentaje del total corresponde a cada status
- Colores: COMPLETADA=verde, PENDIENTE=amarillo, FALLIDA=rojo, CANCELADA=morado

**Sección B — Top sucursal destacada**
- Consume el endpoint `/metricas/top-sucursal` que hiciste en la tarea de la sesión 9
- Muéstrala con `st.success()` como un mensaje destacado:
  > Sucursal líder: **CDMX-Norte** con $12,345.67 en ventas

**Entrega:** El archivo `dashboard.py` modificado con las dos secciones funcionando.
Toma un screenshot del dashboard completo con las nuevas secciones visibles.

> La tarea de esta sesión y la de la sesión 9 se conectan: el endpoint que hiciste ahí lo consumes aquí. Así funciona un sistema real — cada capa usa la anterior.